# 歷史版本 A：最早期草稿（5/19–5/22）

**用途**：最初的 PCA 與分群嘗試，欄位使用中文名稱，未排除連江縣。

| 項目 | 說明 |
|---|---|
| 輸入 | data/raw CSV |
| 輸出 | （歷史探索，圖表為即時輸出） |
| 來源 | MVA 期末報告分析.ipynb（Colab 原檔，191 cells） 之 cell 0–27 |

> 歷史版本，保留供追溯；不保證可獨立執行，最終流程請見 ../final_analysis/ 與 ../../src/。


In [ ]:
# 共用設定：載入資料並提供 df / pivot_table_age_std / pivot_table_cleaned / scaled_data / taiwan_map / prop 等變數
# （取代原 Colab 的 Drive 掛載、字型下載與 GDAL 設定；詳見 _legacy_setup.py）
import sys; sys.path.insert(0, '..')   # _legacy_setup.py 位於 notebooks/
from _legacy_setup import *
df = pd.read_csv(CSV_PATH)  # 早期版本使用原始中文欄位名稱（覆蓋共用設定的英文欄位版）

可以直接跳到 [Update 05/29](https://colab.research.google.com/drive/1M4_KBoDEZ0BYlI7N2MswYAXsc5jGKFyx#scrollTo=dyDTarFIUj6U&line=3&uniqifier=1) 直接執行

In [ ]:
df_filtered = df[(df['性別'] == '全') &(df['縣市別'] != '全國') & (df['癌症別'] == '肺、支氣管及氣管')]
display(df_filtered)

In [ ]:
pivot_table_age_std = df_filtered.pivot_table(
    index='縣市別',
    columns='癌症診斷年',
    values='年齡標準化發生率  WHO 2000世界標準人口 (每10萬人口)',
    aggfunc='mean' # Assuming you want the mean if there are multiple entries
)
display(pivot_table_age_std)

In [ ]:
correlation_matrix = pivot_table_age_std.corr()
display(correlation_matrix.head())

Now, let's use hierarchical clustering to reorder the correlation matrix for better visualization in the heatmap.

In [ ]:
import scipy.cluster.hierarchy as sch
import matplotlib.pyplot as plt
import seaborn as sns

# Perform hierarchical clustering
linked = sch.linkage(correlation_matrix, method='ward')

# Get the order of the clusters
cluster_order = sch.dendrogram(linked, no_plot=True)['leaves']

# Reorder the correlation matrix based on the clustering result
reordered_correlation_matrix = correlation_matrix.iloc[cluster_order, cluster_order]
display(reordered_correlation_matrix.head())

Finally, let's visualize the reordered correlation matrix using a heatmap.

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(reordered_correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Age-Standardized Incidence Rates (Ordered by Hclust)')
plt.show()

## Principal Component Analysis (PCA) on Correlation Matrix

Now, let's perform PCA on the `correlation_matrix` to reduce dimensionality and identify underlying patterns. We'll start by importing `PCA` from `sklearn.decomposition` and fitting it to our data.

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

# Initialize PCA. We will keep all components for the scree plot first.
pca = PCA()
pca.fit(correlation_matrix)

# Explained variance ratio
explained_variance_ratio = pca.explained_variance_ratio_

# Cumulative explained variance
cumulative_explained_variance = np.cumsum(explained_variance_ratio)

print("Explained Variance Ratio:")
print(explained_variance_ratio)
print("\nCumulative Explained Variance:")
print(cumulative_explained_variance)

### Scree Plot

A scree plot helps visualize the eigenvalues (explained variance) for each principal component, allowing us to determine the optimal number of components to retain.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio, marker='o', linestyle='--')
plt.title('Scree Plot')
plt.xlabel('Number of Components')
plt.ylabel('Explained Variance Ratio')
plt.grid(True)
plt.show()

### Cumulative Explained Variance Plot

This plot shows the cumulative sum of explained variance, which helps in deciding how many principal components are needed to explain a certain percentage of the total variance.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cumulative_explained_variance) + 1), cumulative_explained_variance, marker='o', linestyle='--')
plt.axhline(y=0.8, color='r', linestyle='-', label='80% Cumulative Variance')
plt.title('Cumulative Explained Variance Plot')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance Ratio')
plt.grid(True)
plt.legend()
plt.show()

### Biplot

A biplot visualizes the relationship between the observations (rows) and the variables (columns) in the PCA space. It helps in understanding how original variables contribute to the principal components and how observations are grouped.

In [ ]:
# For biplot, we usually consider the first two principal components
pca_2_components = PCA(n_components=2)
components = pca_2_components.fit_transform(correlation_matrix)
loadings = pca_2_components.components_.T * np.sqrt(pca_2_components.explained_variance_)

fig, ax = plt.subplots(figsize=(12, 12))
ax.scatter(components[:, 0], components[:, 1], alpha=0.5)

for i, feature in enumerate(correlation_matrix.columns):
    ax.arrow(0, 0, loadings[i, 0], loadings[i, 1], color='r', alpha=0.8)
    ax.text(loadings[i, 0] * 1.15, loadings[i, 1] * 1.15, feature, color='g', ha='center', va='center')

ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
ax.set_title('Biplot of PCA on Correlation Matrix')
ax.grid(True)
plt.show()

## Cluster Analysis

We will now perform cluster analysis on the `pivot_table_age_std` to group similar `縣市別` (cities/counties) based on their age-standardized incidence rates over the years. We'll test different distance measures and clustering algorithms as requested.

### Data Preparation for Clustering

First, we need to handle any missing values in `pivot_table_age_std` and scale the data. Clustering algorithms are sensitive to missing values and the scale of features.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Ensure the data is numeric and handle potential NaNs
# For clustering, we need to ensure no NaNs. Let's impute with the mean of each column (year).
clustering_data = pivot_table_age_std.copy()

# Impute NaNs with the mean of each column
for col in clustering_data.columns:
    if clustering_data[col].isnull().any():
        clustering_data[col] = clustering_data[col].fillna(clustering_data[col].mean())

# Standardize the data. We want to cluster the rows (縣市別).
# So we scale features (years) for each observation (縣市別).
scaler = StandardScaler()
scaled_clustering_data = scaler.fit_transform(clustering_data)

# Convert back to DataFrame for easier inspection, keeping original index and columns
scaled_clustering_df = pd.DataFrame(scaled_clustering_data, index=clustering_data.index, columns=clustering_data.columns)
display(scaled_clustering_df.head())


### K-Means Clustering

We'll apply K-Means clustering. For demonstration, we'll choose a number of clusters (e.g., `k=4`). To visualize the clusters, we'll reduce the dimensionality using PCA to two components.

In [ ]:
from sklearn.decomposition import PCA

k = 4  # Let's start with 4 clusters for demonstration
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10) # n_init is set to 'auto' or explicit int in newer versions
kmeans_labels = kmeans.fit_predict(scaled_clustering_data)

# Add cluster labels to the original DataFrame
clustering_data['KMeans_Cluster'] = kmeans_labels
display(clustering_data.head())

# Visualize K-Means clusters using PCA for dimensionality reduction
pca_kmeans = PCA(n_components=2)
components_kmeans = pca_kmeans.fit_transform(scaled_clustering_data)

plt.figure(figsize=(10, 8))
sns.scatterplot(
    x=components_kmeans[:, 0],
    y=components_kmeans[:, 1],
    hue=kmeans_labels,
    palette='viridis',
    s=100,
    alpha=0.8
)
plt.title(f'K-Means Clustering (k={k}) - PCA Reduced')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True)
plt.show()

### Hierarchical Clustering

Now, let's explore hierarchical clustering using different distance metrics and linkage methods. We'll use Pearson Correlation and Spearman Correlation as distance measures, and 'single', 'complete', and 'average' as linkage methods. For each combination, we'll display a dendrogram.

#### Hierarchical Clustering with Pearson Correlation Distance

#### Hierarchical Clustering with Spearman Correlation Distance

In [ ]:
from scipy.stats import rankdata

# Define a custom function for Spearman correlation distance
def spearman_correlation_distance(u, v):
    # Calculate ranks for each vector u and v
    rank_u = rankdata(u)
    rank_v = rankdata(v)
    # Calculate Pearson correlation coefficient between the ranks
    corr_matrix = np.corrcoef(rank_u, rank_v)
    corr = corr_matrix[0, 1]

    # Handle NaN correlation (e.g., if one or both vectors are constant)
    if np.isnan(corr):
        # A common practice is to assign a neutral distance of 1.0 (equivalent to correlation 0)
        return 1.0
    return 1 - corr

# Calculate Spearman correlation-based distance using the custom metric
spearman_distances_condensed = pdist(scaled_clustering_data, metric=spearman_correlation_distance)

plt.figure(figsize=(18, 12))

for i, method in enumerate(linkage_methods):
    plt.subplot(3, 1, i + 1)
    linked = linkage(spearman_distances_condensed, method=method)
    dendrogram(linked,
               orientation='top',
               labels=clustering_data.index.tolist(),
               distance_sort='descending',
               show_leaf_counts=True,
               color_threshold=0.7 * max(linked[:,2])) # Adjust threshold for better visibility
    plt.title(f'Hierarchical Clustering (Spearman - {method} Linkage)')
    plt.xlabel('縣市別')
    plt.ylabel('Distance')

plt.tight_layout()
plt.show()